# Build a RAG pipeline in Python over arXiv papers with Infino

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline over real arXiv research papers using [Infino](https://pypi.org/project/infino/) — a single engine that stores your data as Apache Parquet on local disk or object storage and runs **SQL, full-text (BM25), and vector search** over it. No separate vector database to run or keep in sync.

**What you'll do:**
1. Pull a sample of real arXiv ML papers from the HuggingFace Hub.
2. Chunk and embed them locally (no API key).
3. Index them in an Infino table that lives on local disk.
4. Retrieve the most relevant chunks for a question with vector search.
5. Assemble a grounded prompt and (optionally) generate an answer with an LLM.

Everything except the optional final answer runs locally and key-free — copy, paste, run.

## Setup

Install the dependencies (skip if you already ran `pip install -r requirements.txt`):

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.chunking import chunk_text
from _shared.datasets import load_arxiv

/Users/pratyushlokhande/InfinoAI/workspace/infino/infino-python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load real arXiv papers

We stream a small sample of real arXiv ML papers (title + abstract) from the HuggingFace Hub. Bump `n` to index more.

In [3]:
papers = load_arxiv(n=200)
print(f"loaded {len(papers)} papers")
print(papers[0]["title"])
print(papers[0]["abstract"][:300], "...")

loaded 200 papers
Learning from compressed observations
The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to appro ...


## 2. Chunk and embed

Each abstract is split into overlapping word windows, and every chunk is embedded locally with `all-MiniLM-L6-v2` (384-dim, normalized). We keep the paper title alongside each chunk so we can cite the source.

In [4]:
titles, texts = [], []
for p in papers:
    for chunk in chunk_text(p["abstract"]):
        titles.append(p["title"])
        texts.append(chunk)

print(f"{len(texts)} chunks from {len(papers)} papers")
embeddings = embed(texts)  # local; first call downloads the small model

347 chunks from 200 papers


## 3. Index in Infino (local disk)

We connect to a **local** catalog (a directory — no server, no S3 setup), declare a schema, and index two columns: `text` for full-text search and `emb` for vector search. The `_id` column is added automatically.

One `append` is one atomic commit.

In [5]:
DB_DIR = "./arxiv_rag_data"
shutil.rmtree(DB_DIR, ignore_errors=True)  # fresh run each time

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("title", pa.large_utf8(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = infino.IndexSpec().fts("text").vector("emb", DIM, n_cent=256, metric=METRIC)
docs = db.create_table("papers", schema, spec)

docs.append(pa.record_batch([
    pa.array(titles, type=pa.large_utf8()),
    pa.array(texts, type=pa.large_utf8()),
    as_vector_column(embeddings),
], schema=schema))

n = db.query_sql("SELECT COUNT(*) AS n FROM papers").to_pydict()["n"][0]
print(f"indexed {n} chunks")

indexed 347 chunks


## 4. Retrieve

Embed the question and run vector k-NN. With a cosine index, a **smaller score is nearer**. We project the columns we want back — `title`, `text`, and `score`.

In [6]:
def retrieve(question: str, k: int = 4):
    res = docs.vector_search(
        "emb", embed_query(question), k=k,
        projection=["title", "text", "score"],
    ).to_pydict()
    return list(zip(res["title"], res["text"], res["score"]))


question = "How do neural networks learn representations from data?"
hits = retrieve(question)
for title, text, score in hits:
    print(f"[{score:.3f}] {title[:70]}\n  {text[:160]}...\n")

[0.569] Automatic Pattern Classification by Unsupervised Learning Using
  Dime
  presented is reproduced at the output by back propagating the error. The mirroring neural network is capable of reducing the input vector to a great degree (app...

[0.595] When is there a representer theorem? Vector versus matrix regularizers
  We consider a general class of regularization methods which learn a vector of parameters on the basis of linear measurements. It is well known that if the regul...

[0.617] Statistical Mechanics of Nonlinear On-line Learning for Ensemble
  Tea
  We analyze the generalization performance of a student in a model composed of nonlinear perceptrons: a true teacher, ensemble teachers, and the student. We calc...

[0.624] Positive factor networks: A graphical framework for modeling
  non-neg
  learning algorithms that leverage existing NMF algorithms and that are straightforward to implement. We present a target tracking example and provide results fo...



## 5. Generate a grounded answer

Assemble the retrieved chunks into a prompt and answer it. The `_shared.llm` helper uses **Azure AI Foundry** when `AZURE_AI_ENDPOINT` / `AZURE_AI_API_KEY` are set (read from a local `.azure.env`), falls back to `OPENAI_API_KEY`, and returns `None` if neither is configured — so the notebook always runs, printing the grounded context when there is no LLM.

In [7]:
from _shared.llm import complete

context = "\n\n".join(f"- {title}: {text}" for title, text, _ in hits)
prompt = (
    "Answer the question using only the context below. Cite paper titles.\n\n"
    f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

reply = complete(prompt)
if reply:
    print(reply)
else:
    print("No LLM configured — grounded context that would be sent to the model:\n")
    print(prompt)

Neural networks learn representations by transforming input data through internal layers so that useful features emerge for the task.

From **“Automatic Pattern Classification by Unsupervised Learning Using Dimensionality Reduction of Data with Mirroring Neural Networks”**, a mirroring neural network learns a compressed internal code by reproducing the input at the output. In this setup, the central hidden layer becomes a reduced representation of the data—about **1/30th of the original size**—while still allowing reconstruction of the input. These learned features can then be used for classification, for example with Forgy’s algorithm.

From **“Positive factor networks: A graphical framework for modeling non-negative sequential data”**, representations are learned as **non-negative factors** that explain sequential observations. This gives structured latent representations useful for tasks like tracking, music transcription, source separation, and speech recognition.

From **“Statisti

## What just happened

One Infino table on local disk held the paper text **and** its vector index — no separate vector database. The same table is also full-text searchable (`docs.bm25_search(...)`) and queryable with SQL (`db.query_sql(...)`) over the exact same rows.

**Next:** [`02_hybrid_rag.ipynb`](02_hybrid_rag.ipynb) combines keyword and vector search in a single query for higher recall.